## DONE HARD SPELLCHECKING AND NOT DONE REMOVING "said" PHRASE

In [1]:
import pandas as pd

In [2]:
test_df = pd.read_csv("test.csv")
test_df.head()

,Source,context,reply,level_1_class,toxicity_type,context_en,reply_en
0,news portal,চট্টগ্রাম বাদে সারাদেশে যথাসময়ে চলবে এইচএসসি ...,সঠিক সিদ্ধান্ত,Non-toxic,NaN,HSC exam will be conducted on time in the enti...,right decision
1,news portal,চট্টগ্রাম বাদে সারাদেশে যথাসময়ে চলবে এইচএসসি ...,একদম ঠিক করছেন,Non-toxic,NaN,HSC exam will be conducted on time in the enti...,You are absolutely right
2,news portal,চট্টগ্রাম বাদে সারাদেশে যথাসময়ে চলবে এইচএসসি ...,ওরা আস্থা বেয়াদব কোন পদার্থের চলবে না ওদের কথ...,Toxic,Insult,HSC exam will be conducted on time in the enti...,ওরা আস্থা বেয়াদব কোন পদার্থের চলবে না ওদের কথ...
3,news portal,কেন পেছানো হলো না পরীক্ষা? এমপি রুমিনের প্রশ্ন...,"যে শিক্ষামন্ত্রী পানির ভেতরে পরীক্ষা নেয়, তাক...",Toxic,Mockery / Ridicule,Why was the test not delayed? Education Minist...,What is the name of the education minister who...
4,news portal,কেন পেছানো হলো না পরীক্ষা? এমপি রুমিনের প্রশ্ন...,কি জিজ্ঞেস করা হল আর কি উত্তর দিলে,Non-toxic,NaN,Why was the test not delayed? Education Minist...,What was asked and what was the answer?


In [3]:
test_df.drop(columns=["context", "reply"], inplace=True)

In [4]:
test_df.rename(columns={"context_en": "context", "reply_en": "reply"}, inplace=True)

In [5]:
test_df.head()

,Source,level_1_class,toxicity_type,context,reply
0,news portal,Non-toxic,NaN,HSC exam will be conducted on time in the enti...,right decision
1,news portal,Non-toxic,NaN,HSC exam will be conducted on time in the enti...,You are absolutely right
2,news portal,Toxic,Insult,HSC exam will be conducted on time in the enti...,ওরা আস্থা বেয়াদব কোন পদার্থের চলবে না ওদের কথ...
3,news portal,Toxic,Mockery / Ridicule,Why was the test not delayed? Education Minist...,What is the name of the education minister who...
4,news portal,Non-toxic,NaN,Why was the test not delayed? Education Minist...,What was asked and what was the answer?


In [6]:
test_df["toxicity_type"] = test_df["toxicity_type"].fillna("None")
test_df["toxicity_type"].value_counts()

toxicity_type
None                                   1041
Mockery / Ridicule                      468
Insult                                  437
Harassment                              112
Profanity                                80
Sexual Harassment / Objectification      73
Hate Speech                              30
Incitement / Encouragement of Harm       28
Dehumanization                           25
Identity Attack                          15
Toxic Endorsement / Agreement            12
Threat                                    8
Name: count, dtype: int64

In [7]:
import re
bengali_letters = re.compile(r"[\u0980-\u09FF]")

banglish_words = set("""tumi ami tui tora amra apni amar tomar kemon keno kothay bhalo valo
nai hobe korbo korchi ekta ekhon eta ota lagbe bolo naki tahole shala shuor kutta
bokachoda pagol gadha harami tomake amake onek kichu jodi""".split())

def is_not_english(text):
  # 1) Bengali letters
  if bengali_letters.search(text):
    return True
  # 2) any letter outside the Latin alphabet (code > 591 means "not a Latin letter")
  if any(ch.isalpha() and ord(ch) > 591 for ch in text):
    return True
  # 3) romanized Bengali: 2 or more Bengali words
  words = re.findall(r"[a-z]+", text.lower())
  if sum(w in banglish_words for w in words) >= 2:
    return True
  return False

bad_prev   = test_df["context"].apply(is_not_english)
bad_target = test_df["reply"].apply(is_not_english)
remove = bad_prev | bad_target          # remove the row if EITHER column is not English

print("Rows with non-English text:", remove.sum())
test_df[remove].to_csv("removed_non_english_rows_from_test_data.csv", index=False)   # keep a copy to check what was removed
test_df = test_df[~remove].reset_index(drop=True)
print("Rows left:", len(test_df))

Rows with non-English text: 522
Rows left: 1807


In [8]:
before = len(test_df)
test_df = test_df.drop_duplicates(subset=["context", "reply"]).reset_index(drop=True)    ## Only drop hen the values in both the columns are same
print("Duplicate rows removed:", before - len(test_df))
print("Rows left:", len(test_df))

Duplicate rows removed: 3
Rows left: 1804


In [9]:
import re
import emoji
from spellchecker import SpellChecker

# Initialize the spell checker once outside the function for speed
spell = SpellChecker()

short_forms = {
  "won't": "will not", "can't": "can not", "cannot": "can not", "ain't": "is not",
  "let's": "let us", "that's": "that is", "it's": "it is", "he's": "he is", "she's": "she is",
  "there's": "there is", "what's": "what is", "who's": "who is",
  "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am", "'d": " would",
}

def clean_text_with_spellcheck(text):
  text = str(text)
  text = text.replace("\u2014", " ").replace("\u2013", " ")       # long dashes -> space
  text = text.replace("\u2019", "'").replace("\u2018", "'")       # curly apostrophes -> normal '
  
  # emoji -> words
  text = emoji.demojize(text, delimiters=(" ", " ")) 
  text = text.lower()
  
  # 1. FIXED REGEX: Collapse 3+ repeated characters down to 2 ONLY for letters (protects numbers like 1000)
  text = re.sub(r"([a-zA-Z])\1{2,}", r"\1\1", text)               # sooooo -> soo
  
  # 2. Expand short forms
  for short, full in short_forms.items():                          # don't -> do not
    text = text.replace(short, full)
  
  text = text.replace("'s", "")                                    # remove possessive: hasina's -> hasina
  
  # 3. SPELL CHECKER: Fix remaining double letters (soo -> so, loove -> love)
  words = text.split()
  corrected_words = []
  for word in words:
    # Check if the word is misspelled and has a double letter
    if spell.unknown([word]) and re.search(r"([a-z])\1", word):
      # Try reducing the double letter to a single letter
      reduced_word = re.sub(r"([a-z])\1", r"\1", word)
      # If the reduced word exists in the dictionary, use it
      if not spell.unknown([reduced_word]):
        word = reduced_word
    corrected_words.append(word)
  text = " ".join(corrected_words)

  # 4. Final punctuation and spacing cleanup
  text = re.sub(r"[^a-z0-9!?_ ]", " ", text)                        # keep only letters, numbers, ! and ?
  text = re.sub(r"([!?])", r" \1 ", text)                          # put spaces around ! and ?
  text = re.sub(r"\s+", " ", text).strip()                         # remove extra spaces
  return text

# Apply the updated function to your dataframe
test_df["context_clean"]   = test_df["context"].apply(clean_text_with_spellcheck)
test_df["reply_clean"] = test_df["reply"].apply(clean_text_with_spellcheck)

# Look at a few examples: before and after
test_df[["reply", "reply_clean"]].sample(5, random_state=1)


,reply,reply_clean
1557,"Is Masud your illegitimate father, Razakar's c...",is masud your illegitimate father razakar chil...
962,right brother,right brother
705,What does that mean? Kaka Myader had been stay...,what does that mean ? kaka myader had been sta...
573,Brokerage all life Brokerage all life but not ...,brokerage all life brokerage all life but not ...
1429,Brokers in India think so,brokers in india think so


In [10]:
label_to_id = {'None': 0, 
               'Dehumanization': 1, 
               'Harassment': 2, 
               'Hate Speech': 3, 
               'Identity Attack': 4, 
               'Incitement / Encouragement of Harm': 5,
               'Insult': 6, 
               'Mockery / Ridicule': 7, 
               'Profanity': 8, 
               'Sexual Harassment / Objectification': 9, 
               'Threat': 10, 
               'Toxic Endorsement / Agreement': 11
              }

test_df["label"]    = test_df["toxicity_type"]
test_df["label_id"] = test_df["label"].map(label_to_id)
test_df["is_toxic"] = (test_df["label"] != "None").astype(int)

print(label_to_id)
test_df["label"].value_counts()

{'None': 0, 'Dehumanization': 1, 'Harassment': 2, 'Hate Speech': 3, 'Identity Attack': 4, 'Incitement / Encouragement of Harm': 5, 'Insult': 6, 'Mockery / Ridicule': 7, 'Profanity': 8, 'Sexual Harassment / Objectification': 9, 'Threat': 10, 'Toxic Endorsement / Agreement': 11}


label
None                                   786
Mockery / Ridicule                     375
Insult                                 346
Harassment                              88
Profanity                               57
Sexual Harassment / Objectification     55
Hate Speech                             26
Incitement / Encouragement of Harm      21
Dehumanization                          20
Identity Attack                         13
Toxic Endorsement / Agreement           10
Threat                                   7
Name: count, dtype: int64

In [11]:
columns_to_save = ["context", "reply",   # original text (use for BERT)
                   "context_clean", "reply_clean",                 # cleaned text (use for TF-IDF / LSTM)
                   "label", "label_id", "is_toxic",              # labels
                  ]

test_df[columns_to_save].to_csv("test_data_with_preprocessing1.csv", index=False)
print("Saved clean_data.csv with", len(test_df), "rows")
test_df[columns_to_save].head()

Saved clean_data.csv with 1804 rows


,context,reply,context_clean,reply_clean,label,label_id,is_toxic
0,HSC exam will be conducted on time in the enti...,right decision,hsc exam will be conducted on time in the enti...,right decision,None,0,0
1,HSC exam will be conducted on time in the enti...,You are absolutely right,hsc exam will be conducted on time in the enti...,you are absolutely right,None,0,0
2,Why was the test not delayed? Education Minist...,What is the name of the education minister who...,why was the test not delayed ? education minis...,what is the name of the education minister who...,Mockery / Ridicule,7,1
3,Why was the test not delayed? Education Minist...,What was asked and what was the answer?,why was the test not delayed ? education minis...,what was asked and what was the answer ?,None,0,0
4,Why was the test not delayed? Education Minist...,Helicopter meets Water meets,why was the test not delayed ? education minis...,helicopter meets water meets,Mockery / Ridicule,7,1
